# Milestone 2 — MERFISH receptor heatmaps

Mean log2(CPM+1) per **cell type** × **brain area** (CCF parcellation), driven by `receptor_query_config.yaml`.

Unlike scRNA (notebook 01), MERFISH resolves config sub-regions (e.g. VISp vs VISpm) at single-cell level.
Genes outside the ~500-gene panel use the imputed matrix when `use_imputed_merfish: true` (~50 GB on first access).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import pandas as pd

from src.config import DEFAULT_OUTPUT_DIR, load_config, start_run
from src.data_loaders import (
    get_abc_cache,
    load_merfish_cell_metadata,
    check_merfish_genes,
    load_merfish_expression_subset,
    aggregate_merfish_expression,
    family_gene_region_matrix_merfish,
    combined_heatmap_matrix,
)
from src.config import restrict_config_to_genes
from src.plotting import plot_family_heatmap, plot_combined_heatmap

In [ ]:
EXPLORATION_ROOT = DEFAULT_OUTPUT_DIR

CONFIG_PATH = PROJECT_ROOT / "receptor_query_config.yaml"
config = load_config(CONFIG_PATH)
config["_dataset_modality"] = "merfish"

OUTPUT_DIR = start_run(
    PROJECT_ROOT,
    config,
    dataset=config["data"]["merfish_dataset"],
    exploration_root=EXPLORATION_ROOT,
    notebook="02_merfish_heatmaps",
)
assert not str(OUTPUT_DIR).startswith(str(PROJECT_ROOT)), (
    f"OUTPUT_DIR must be outside the repo; got {OUTPUT_DIR}"
)

print(f"Genes: {len(config['_all_genes'])}")
print(f"Brain areas: {config['brain_areas']}")
print(f"Cell type level: {config['cell_type_level']}")
filt = config.get("cell_type_name_filter") or []
print(f"Cell type name filter: {filt if filt else '(none — all types)'}")
print(f"Run dir: {OUTPUT_DIR}")
print(f"Manifest: {OUTPUT_DIR / 'run_manifest.json'}")

In [ ]:
cache = get_abc_cache(config)
print("Manifest:", cache.current_manifest)

In [ ]:
cell_meta = load_merfish_cell_metadata(cache, config)
print(f"Filtered cells: {len(cell_meta):,}")
print("\nCells per brain_area (CCF parcellation):")
print(cell_meta.groupby("brain_area").size().sort_values(ascending=False))

In [ ]:
requested = list(config["_all_genes"])
genes_flat_orig = dict(config["_genes_flat"])
availability = check_merfish_genes(cache, requested, config)

print(f"Measured ({len(availability['measured'])}): {availability['measured'][:10]}{'...' if len(availability['measured']) > 10 else ''}")
print(f"Imputed ({len(availability['imputed'])}): {availability['imputed'][:10]}{'...' if len(availability['imputed']) > 10 else ''}")
if availability["missing"]:
    print(f"\nMissing ({len(availability['missing'])}):")
    for gene in availability["missing"]:
        print(f"  {gene} ({genes_flat_orig.get(gene, 'unknown')})")

loadable = availability["measured"] + availability["imputed"]
if not loadable:
    raise RuntimeError("No requested genes available in MERFISH measured or imputed panels.")

adata = load_merfish_expression_subset(cache, loadable, cell_meta, config)
if adata is None:
    raise RuntimeError("No expression data loaded; check cache downloads.")

loaded_genes = adata.var["gene_symbol"].tolist()
restrict_config_to_genes(config, loaded_genes)

print(f"\nProceeding with {len(config['_all_genes'])} / {len(requested)} genes.")
print(f"Families with data: {config['_families']}")
if "source" in adata.var.columns:
    print(adata.var["source"].value_counts())
print(adata)

In [ ]:
agg_long = aggregate_merfish_expression(adata, cell_meta, config)
print(agg_long.head())
print(f"\nAggregated rows: {len(agg_long):,}")

In [ ]:
print(f"Saving figures to {OUTPUT_DIR}")

for family in config["_families"]:
    mat = family_gene_region_matrix_merfish(agg_long, family, config)
    if mat.empty:
        warnings.warn(f"No data for family {family!r}; skipping heatmap.")
        continue
    path = plot_family_heatmap(family, mat, config, output_dir=OUTPUT_DIR)
    print(f"Saved {path}")

In [ ]:
combined = combined_heatmap_matrix(agg_long, config)
print(f"Combined heatmap: {combined.shape[0]} cell types × {combined.shape[1]} genes")
path = plot_combined_heatmap(combined, config, output_dir=OUTPUT_DIR)
print(f"Saved {path}")

In [ ]:
if config["output"].get("save_processed_data", True):
    parquet_path = OUTPUT_DIR / "aggregated_merfish.parquet"
    agg_long.to_parquet(parquet_path, index=False)
    print(f"Saved aggregated matrix to {parquet_path}")